In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Importation des bibliotheques 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from PIL import Image
import os
import numpy as np

# Function to perform EDA
def perform_eda(dataset_path):
    class_names = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
    print(f"Number of classes: {len(class_names)}")
    print(f"Classes: {class_names}")

    # Count images per class
    class_counts = {}
    for class_name in class_names:
        class_path = os.path.join(dataset_path, class_name)
        class_counts[class_name] = len([f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))])
    
    print(f"Class distribution: {class_counts}")

    # Plot class distribution
    plt.figure(figsize=(10, 5))
    sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()), palette="viridis")
    plt.title("Number of Images per Class")
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45)
    plt.show()

    # Display sample images
    print("Displaying sample images from each class:")
    fig, axes = plt.subplots(1, len(class_names), figsize=(15, 5))
    for i, class_name in enumerate(class_names):
        class_path = os.path.join(dataset_path, class_name)
        sample_image = os.path.join(class_path, os.listdir(class_path)[0])
        img = Image.open(sample_image)
        axes[i].imshow(img)
        axes[i].set_title(class_name)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

    # Check for image size consistency
    print("Analyzing image dimensions:")
    image_sizes = []
    for class_name in class_names:
        class_path = os.path.join(dataset_path, class_name)
        for filename in os.listdir(class_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                img_path = os.path.join(class_path, filename)
                with Image.open(img_path) as img:
                    image_sizes.append(img.size)

    image_sizes = np.array(image_sizes)
    print(f"Unique image dimensions: {np.unique(image_sizes, axis=0)}")

    # Pixel intensity distribution
    print("Analyzing pixel intensity distribution:")
    pixel_values = []
    for class_name in class_names:
        class_path = os.path.join(dataset_path, class_name)
        for filename in os.listdir(class_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                img_path = os.path.join(class_path, filename)
                with Image.open(img_path).convert("L") as img:  # Convert to grayscale for simplicity
                    pixel_values.extend(np.array(img).flatten())
    
    plt.figure(figsize=(10, 5))
    sns.histplot(pixel_values, bins=50, kde=True, color='blue')
    plt.title("Pixel Intensity Distribution")
    plt.xlabel("Pixel Value")
    plt.ylabel("Frequency")
    plt.show()

# Perform EDA
dataset_path = '/kaggle/input/thermal-images-of-induction-motor'
perform_eda(dataset_path)


In [ ]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image
import matplotlib.pyplot as plt


In [ ]:
IMAGE_SIZE = (64, 64)

Function to load and preprocess images


In [ ]:
def load_and_preprocess_images(dataset_path):
    images = []
    class_names = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])  
    
    for class_name in class_names:
        class_path = os.path.join(dataset_path, class_name)
       
        for filename in os.listdir(class_path):
            img_path = os.path.join(class_path, filename)
            if not filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                continue
            try:
                img = Image.open(img_path).convert('RGB')  # Convert to RGB
                img = img.resize(IMAGE_SIZE)
                img_array = np.array(img).astype('float32')
                img_array = (img_array - 127.5) / 127.5  # Normalize to [-1, 1]
                images.append(img_array)
            except Exception as e:
                print(f"Error loading image {filename}: {e}")
                continue
    
    return np.array(images)


Load and preprocess images from the dataset


In [ ]:
dataset_path = '/kaggle/input/thermal-images-of-induction-motor'
train_images = load_and_preprocess_images(dataset_path)

data augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

augmented_images = []
for image in train_images:
    image = np.expand_dims(image, axis=0)  
    for _ in range(10):  # Generate 10 augmented versions per image
        augmented_images.append(datagen.random_transform(image[0]))

train_images = np.array(augmented_images)
print(train_images.shape)

GAN model definition

In [ ]:
class GAN(tf.keras.Model):
    def __init__(self, generator, discriminator):
        super(GAN, self).__init__()
        self.generator = generator
        self.discriminator = discriminator
    
    def compile(self, g_optimizer, d_optimizer, loss_fn):
        super(GAN, self).compile()
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.loss_fn = loss_fn

    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        noise = tf.random.normal([batch_size, self.generator.input_shape[-1]])

        # Train the discriminator
        with tf.GradientTape() as d_tape:
            generated_images = self.generator(noise, training=True)
            real_output = self.discriminator(real_images, training=True)
            fake_output = self.discriminator(generated_images, training=True)
            d_loss_real = self.loss_fn(tf.ones_like(real_output), real_output)
            d_loss_fake = self.loss_fn(tf.zeros_like(fake_output), fake_output)
            d_loss = d_loss_real + d_loss_fake

        d_gradients = d_tape.gradient(d_loss, self.discriminator.trainable_variables)
        self.d_optimizer.apply_gradients(zip(d_gradients, self.discriminator.trainable_variables))
        
        # Train the generator
        with tf.GradientTape() as g_tape:
            generated_images = self.generator(noise, training=True)
            fake_output = self.discriminator(generated_images, training=True)
            g_loss = self.loss_fn(tf.ones_like(fake_output), fake_output)

        g_gradients = g_tape.gradient(g_loss, self.generator.trainable_variables)
        self.g_optimizer.apply_gradients(zip(g_gradients, self.generator.trainable_variables))

        return {"d_loss": d_loss, "g_loss": g_loss}


In [ ]:
noise_dim = 100

Build the generator model

In [ ]:
def build_generator(noise_dim):
    model = models.Sequential()

    model.add(layers.Input(shape=(noise_dim,)))
    model.add(layers.Dense(8 * 8 * 256, use_bias=False))  # Latent vector into initial dense layer
    model.add(layers.BatchNormalization())  
    model.add(layers.ReLU())  
    model.add(layers.Reshape((8, 8, 256)))  # Reshape to 8x8x256 feature map
    
    # First upsampling layer: Output size will be (16, 16, 128)
    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    # Second upsampling layer: Output size will be (32, 32, 64)
    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    # Third upsampling layer: Output size will be (64, 64, 3) (Final size)
    model.add(layers.Conv2DTranspose(3, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))  

    return model


Build the discriminator model

In [ ]:
def build_discriminator(input_shape=(64, 64, 3)):
    model = models.Sequential()

    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=input_shape))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

In [ ]:
generator = build_generator(noise_dim)
discriminator = build_discriminator()

In [ ]:
# Create the GAN model
gan = GAN(generator, discriminator)

# Compile the GAN model
loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=True)
generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)
gan.compile(generator_optimizer, discriminator_optimizer, loss_fn)

Training the GAN

In [ ]:
epochs = 200
batch_size = 32
dataset = tf.data.Dataset.from_tensor_slices(train_images).shuffle(buffer_size=1000).batch(batch_size)

d_losses = []
g_losses = []

for epoch in range(epochs):
    for real_images in dataset:
        losses = gan.train_step(real_images)
        d_losses.append(losses['d_loss'].numpy())
        g_losses.append(losses['g_loss'].numpy())

    # Generate and save images after each epoch (optional)
    noise = tf.random.normal([16, noise_dim])
    generated_images = generator(noise, training=False)

    # Display images
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        img = (generated_images[i].numpy() + 1) / 2  # Rescale to [0, 1] for display
        ax.imshow(img)
        ax.axis('off')
    plt.show()

    print(f"Epoch {epoch+1}/{epochs} completed.")


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Function to compute correlation
def compute_correlation(image1, image2):
    # Flatten the images
    flat_image1 = image1.flatten()
    flat_image2 = image2.flatten()
    # Compute Pearson correlation coefficient
    correlation = np.corrcoef(flat_image1, flat_image2)[0, 1]
    return correlation

# After the GAN training loop, compare generated and original images
# Select one original image randomly
original_image = train_images[np.random.randint(0, len(train_images))]

# Select one generated image from the last epoch
generated_image = generated_images[0].numpy()  # Select the first generated image from the last epoch

# Rescale the images to [0, 1] for visualization
original_image_rescaled = (original_image + 1) / 2
generated_image_rescaled = (generated_image + 1) / 2

# Compute the correlation between the original and generated image
correlation = compute_correlation(original_image_rescaled, generated_image_rescaled)

# Display the images side by side
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(original_image_rescaled)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title(f"Generated Image\nCorrelation: {correlation:.4f}")
plt.imshow(generated_image_rescaled)
plt.axis('off')

plt.tight_layout()
plt.show()
